# Tune distributed matching with `MatchingConfig`

Compare exact and approximate nearest-neighbour search on the same Spark
data. This notebook uses the FAISS extension directly to evaluate just the
search stage of matching. `Matching` uses this extension when it runs on Spark.

- [Choose settings to change](#matching-settings)
- [Measure neighbour agreement and run time](#compare-search)
- [Clean up FAISS index files](#index-cleanup)

<a id="matching-settings"></a>

## 1. Choose settings to change

`FAISS_FIT_MODE="full"` performs exact search across flat partition indexes.
`"shuffle"` groups controls by cluster and routes a query to multiple clusters.
`FAISS_N_PROBES` sets how many clusters to search in that mode. Probing more
clusters can recover missed neighbours and increase compute or shuffle work.
`BUCKET_SIZE` controls how large clusters are split into groups for Spark tasks.

Set the options before fitting and keep them in scope until prediction
finishes. These options are shared within the driver process. This notebook
runs each configuration sequentially and restores the previous settings.
The small input is for demonstration. A single run includes Spark warm-up
and its timing is not a benchmark.

In [1]:
import os

# Local tutorial settings. Set these before importing PySpark or HypEx.
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYARROW_IGNORE_TIMEZONE", "1")

from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
from unittest.mock import patch

import numpy as np
from pyspark.sql import SparkSession

from hypex.config import MatchingConfig
from hypex.dataset import Dataset, FeatureRole, InfoRole, SmallDataset
from hypex.extensions import SparkFaissExtension
from hypex.utils import BackendsEnum, FaissIndexStorage

# Synthetic data preparation. Generate fixed two-dimensional points.
rng = np.random.default_rng(44)
control_points = rng.normal(size=(256, 2)).tolist()
query_points = rng.normal(size=(48, 2)).tolist()
roles = {"x": FeatureRole(float), "y": FeatureRole(float)}

def spark_dataset(points, spark):
    source = spark.createDataFrame(points, schema="x double, y double").coalesce(1)
    return Dataset(roles=roles, data=source, backend=BackendsEnum.spark, session=spark)

<a id="compare-search"></a>

## 2. Measure neighbour agreement and run time

Run exact search once to get a reference. For each shuffle configuration,
measure the share of queries whose nearest neighbour agrees with the
reference. The comparison collects only the small table of query IDs and
predicted neighbours. For real matching analyses, also check covariate
balance and how often the same control is reused.

A temporary checkpoint directory belongs to the Spark session below. FAISS index files use separate directories managed by `FaissIndexStorage`.

In [2]:
from tempfile import TemporaryDirectory


# Configure the local JVM before Spark starts, while keeping error messages visible.
spark_files = TemporaryDirectory(prefix="hypex-spark-")
log_config = Path(spark_files.name) / "log4j2.properties"
log_config.write_text(
    "rootLogger.level = error\n"
    "rootLogger.appenderRef.console.ref = console\n"
    "appender.console.type = Console\n"
    "appender.console.name = console\n"
    "appender.console.target = SYSTEM_ERR\n"
    "appender.console.layout.type = PatternLayout\n"
    "appender.console.layout.pattern = %p %c: %m%n\n",
    encoding="utf-8",
)
spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypEx MatchingConfig")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.extraJavaOptions", f"-Dlog4j.configurationFile={log_config.as_uri()}")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setCheckpointDir(str(Path(spark_files.name) / "checkpoints"))

In [3]:
def search(controls, queries, mode, probes):
    with patch.multiple(MatchingConfig, FAISS_FIT_MODE=mode, FAISS_N_PROBES=probes):
        with SparkFaissExtension(n_neighbors=1) as engine:
            start = perf_counter()
            result = engine.calc(controls, queries)
            try:
                # Local validation of compact predictions. Join by query index,
                # since Spark results can arrive in different row orders.
                rows = result.reset_index().to_records()
            finally:
                result.unpersist()
            elapsed = perf_counter() - start

    query_column, neighbour_column = rows[0].keys()
    neighbours = {row[query_column]: row[neighbour_column] for row in rows}
    return neighbours, elapsed


controls = spark_dataset(control_points, spark)
queries = spark_dataset(query_points, spark)
exact, exact_seconds = search(controls, queries, "full", probes=1)
print(f"Exact search returned {len(exact)} neighbours.")

Exact search returned 48 neighbours.


<a id="index-cleanup"></a>

## 3. Clean up FAISS index files

**Option A · Run a cleanup cell after the estimator.** `full` search uses
local temporary files to distribute partition indexes. After the search has
finished and its results are materialized, run the next cell.
`FaissIndexStorage.cleanup()` removes all directories registered in the
current Python process, so use it after any other search sharing that process
has finished. `Matching.execute()` already calls it on exit. The notebook
calls the lower-level extension directly, so it handles file cleanup itself.

In [4]:
# Inspect only the directory count. Avoid printing machine-specific paths.
index_dirs = [Path(directory) for directory in FaissIndexStorage.LOCAL_DIRS]
FaissIndexStorage.cleanup()
# Check the filesystem directly because cleanup is best-effort.
assert all(not directory.exists() for directory in index_dirs)
print(f"Removed registered index directories: {len(index_dirs)}")

Removed registered index directories: 1


**Option B · Guard a run with `try/except/finally`.** Use this structure when
running searches from a script or job. The `except` block reports a failure
and raises it again. The `finally` block cleans up files after either outcome.
The extension's context manager in `search` separately releases Spark
resources held by that search.

In [5]:
results = [{"mode": "full", "probes": "—", "agreement": 1.0, "seconds": round(exact_seconds, 2)}]
try:
    for probes in (1, 4):
        neighbours, elapsed = search(controls, queries, "shuffle", probes)
        # Compare the same query IDs so missing results cannot inflate agreement.
        assert neighbours.keys() == exact.keys()
        agreement = sum(neighbours[key] == exact[key] for key in exact) / len(exact)
        results.append(
            {"mode": "shuffle", "probes": str(probes), "agreement": round(agreement, 3),
             "seconds": round(elapsed, 2)}
        )
except Exception:
    print("Spark matching failed. Releasing temporary index files before propagating the error.")
    raise
finally:
    FaissIndexStorage.cleanup()

report_roles = {column: InfoRole() for column in results[0]}
display(SmallDataset.from_dict(results, roles=report_roles))

,mode,probes,agreement,seconds
0,full,—,1.000,2.96
1,shuffle,1,0.938,2.55
2,shuffle,4,1.000,0.78


In [6]:
try:
    spark.stop()
finally:
    spark_files.cleanup()

The settings and results above concern the Spark search stage. For configuring
a complete analysis and inspecting its estimates, see the
[matching tutorial](../03_Matching_and_Causal_Inference.ipynb).